# PROJECT 2

## E-Commerce Fraud Risk Detection using Machine Learning

In [ ]:
import os
import random
import numpy as np
import pandas as pd

import matplotlib.pyplot as plt
import seaborn as sns

import plotly.express as px
import plotly.graph_objects as go

from scipy import stats

from sklearn.impute import SimpleImputer
from sklearn.impute import KNNImputer

from sklearn.preprocessing import LabelEncoder
from sklearn.preprocessing import OneHotEncoder
from sklearn.preprocessing import StandardScaler
from sklearn.preprocessing import MinMaxScaler
from sklearn.preprocessing import RobustScaler
from sklearn.preprocessing import PowerTransformer

plt.style.use("ggplot")

In [ ]:
from sklearn.linear_model import LogisticRegression

from sklearn.tree import DecisionTreeClassifier

from sklearn.ensemble import RandomForestClassifier

from sklearn.neighbors import KNeighborsClassifier

from sklearn.naive_bayes import GaussianNB

from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    balanced_accuracy_score,
    matthews_corrcoef,
    confusion_matrix,
    classification_report,
    roc_curve,
    auc
)

In [ ]:
# Configuration

RANDOM_STATE = 42

TARGET = "is_fraud"

pd.set_option("display.max_columns",None)
pd.set_option("display.max_rows",100)

np.random.seed(RANDOM_STATE)
random.seed(RANDOM_STATE)

In [ ]:
df = pd.read_csv("Dataset for Data Analytics - Sheet1.csv")
df.sample(10)

In [ ]:
print("Shape :\n", df.shape)
print("\n")
print("Columns :\n",df.columns.tolist())
print("\n")
print("Data Types :\n",df.dtypes)
print("\n")
print("Memory Usage :\n",df.memory_usage(deep=True))
print("\n")
df.info()
print("\n")
df.describe(include="all").T

### Missing Values

In [ ]:
missing = pd.DataFrame({
    "Missing Values":df.isnull().sum(),
    "Percentage":round(df.isnull().mean()*100,2)
})

missing.sort_values("Percentage",ascending=False)

### Duplicate Records

In [ ]:
duplicates=df.duplicated().sum()

print("Duplicate Records :",duplicates)

### Unique Values

In [ ]:
unique=pd.DataFrame(df.nunique())

unique.columns=["Unique Values"]

unique.sort_values("Unique Values")

### Convert Date

In [ ]:
df["Date"]=pd.to_datetime(df["Date"])

print(df["Date"].head())

In [ ]:
df["Year"]=df["Date"].dt.year
df["Month"]=df["Date"].dt.month
df["Day"]=df["Date"].dt.day
df["Weekday"]=df["Date"].dt.day_name()
df["Quarter"]=df["Date"].dt.quarter
df["WeekOfYear"]=df["Date"].dt.isocalendar().week

df.head()

### Numerical Columns

In [ ]:
num_cols=df.select_dtypes(include=np.number).columns

num_cols

### Categorical Columns

In [ ]:
cat_cols=df.select_dtypes(include="object").columns

cat_cols

### Correlation Heatmap

In [ ]:
plt.figure(figsize=(12,8))

sns.heatmap(df[num_cols].corr(),
            annot=True,
            cmap="coolwarm",
            linewidth=0.5)

plt.title("Correlation Matrix")

plt.show()

### Distribution Plots


In [ ]:
for col in num_cols:
    
    plt.figure(figsize=(8,4))
    
    sns.histplot(df[col],
                 kde=True,
                 color="steelblue")
    
    plt.title(col)
    
    plt.show()

### Boxplots

In [ ]:
for col in num_cols:
    
    plt.figure(figsize=(8,3))
    
    sns.boxplot(x=df[col],
                color="orange")
    
    plt.title(col)
    
    plt.show()

### Countplots

In [ ]:
for col in cat_cols:
    
    plt.figure(figsize=(12,5))
    
    sns.countplot(y=df[col],
                  order=df[col].value_counts().index)
    
    plt.title(col)
    
    plt.show()

### Top Products

In [ ]:
plt.figure(figsize=(12,6))

df["Product"].value_counts().head(10).plot(kind="bar")

plt.title("Top 10 Products")

plt.ylabel("Orders")

plt.show()

### Payment Method Analysis

In [ ]:
plt.figure(figsize=(8,5))

sns.countplot(x="PaymentMethod",
              data=df)

plt.xticks(rotation=45)

plt.title("Payment Method Distribution")

plt.show()

### Monthly Sales Trend

In [ ]:
monthly=df.groupby("Month")["TotalPrice"].sum()

plt.figure(figsize=(10,5))

monthly.plot(marker="o")

plt.title("Monthly Sales Trend")

plt.ylabel("Revenue")

plt.grid()

plt.show()

### Missing Value Treatment

In [ ]:
print("Missing Values Before Treatment")

print(df.isnull().sum())

In [ ]:
# Numerical Columns

num_cols=df.select_dtypes(include=np.number).columns

# Categorical Columns

cat_cols=df.select_dtypes(include="object").columns

# Fill Numerical Columns

for col in num_cols:

    if df[col].isnull().sum()>0:

        df[col].fillna(df[col].median(),inplace=True)

# Fill Categorical Columns

for col in cat_cols:

    if df[col].isnull().sum()>0:

        df[col].fillna(df[col].mode()[0],inplace=True)

In [ ]:
print("Missing Values After Treatment")

print(df.isnull().sum())

### Outlier Detection using IQR

In [ ]:
def detect_outliers_iqr(data,column):

    Q1=data[column].quantile(0.25)

    Q3=data[column].quantile(0.75)

    IQR=Q3-Q1

    lower=Q1-1.5*IQR

    upper=Q3+1.5*IQR

    return data[(data[column]<lower)|(data[column]>upper)]

In [ ]:
for col in ["Quantity","UnitPrice","ItemsInCart","TotalPrice"]:

    outliers=detect_outliers_iqr(df,col)

    print(col,"=",len(outliers))

In [ ]:
from scipy.stats import zscore

z_cols = [
    "Quantity",
    "UnitPrice",
    "ItemsInCart",
    "TotalPrice"
]

z = np.abs(zscore(df[z_cols]))

z_df = pd.DataFrame(z, columns=z_cols)

z_df.head()

In [ ]:
plt.figure(figsize=(12,5))

sns.boxplot(data=df[num_cols])

plt.xticks(rotation=90)

plt.title("Before Outlier Treatment")

plt.show()

In [ ]:
# Fraud Risk Score

risk_score=np.zeros(len(df))

In [ ]:
# High Total Price

risk_score += np.where(

    df["TotalPrice"] >

    df["TotalPrice"].quantile(0.95),

    35,

    0

)

In [ ]:
# Large Cart

risk_score += np.where(

    df["ItemsInCart"] >

    df["ItemsInCart"].quantile(0.90),

    20,

    0

)

In [ ]:
# High Quantity

risk_score += np.where(

    df["Quantity"] >

    df["Quantity"].quantile(0.90),

    15,

    0

)

In [ ]:
# Coupon Used

risk_score += np.where(

    df["CouponCode"].notnull(),

    10,

    0

)

In [ ]:
# Online Payment

risk_score += np.where(

    df["PaymentMethod"].isin(

        ["Credit Card","Debit Card","UPI"]

    ),

    10,

    0

)

In [ ]:
# Duplicate Shipping Address

duplicate_address=df.groupby(

    "ShippingAddress"

)["OrderID"].transform("count")

risk_score += np.where(

    duplicate_address>3,

    10,

    0

)

In [ ]:
# Customer Daily Orders

daily_orders=df.groupby(

    ["CustomerID","Date"]

)["OrderID"].transform("count")

risk_score += np.where(

    daily_orders>2,

    15,

    0

)

In [ ]:
df["FraudRiskScore"]=risk_score

df["FraudRiskScore"].describe()

### Create Target

In [ ]:
df["is_fraud"]=np.where(

    df["FraudRiskScore"]>=50,

    1,

    0

)

df["is_fraud"].value_counts()

### Fraud Distribution

In [ ]:
plt.figure(figsize=(6,5))

sns.countplot(

    x=df["is_fraud"]

)

plt.title("Synthetic Fraud Distribution")

plt.show()

In [ ]:
fraud=df["is_fraud"].value_counts(normalize=True)*100

print(fraud)

### Feature Engineering

In [ ]:
# Average Product Price

df["AveragePrice"]=df["TotalPrice"]/df["Quantity"]

In [ ]:
# High Value Transaction

df["HighValueTransaction"]=np.where(

    df["TotalPrice"]>

    df["TotalPrice"].median(),

    1,

    0

)

In [ ]:
# Weekend Flag

df["WeekendFlag"]=np.where(

    df["Weekday"].isin(

        ["Saturday","Sunday"]

    ),

    1,

    0

)

In [ ]:
# Customer Purchase Count

df["CustomerPurchaseCount"]=df.groupby(

    "CustomerID"

)["OrderID"].transform("count")

In [ ]:
# Customer Total Spending

df["CustomerSpending"]=df.groupby(

    "CustomerID"

)["TotalPrice"].transform("sum")

In [ ]:
# Product Popularity

df["ProductPopularity"]=df.groupby(

    "Product"

)["OrderID"].transform("count")

In [ ]:
# Average Order Value

df["AverageOrderValue"]=df.groupby(

    "CustomerID"

)["TotalPrice"].transform("mean")

In [ ]:
# Customer Risk Category

conditions=[

    df["FraudRiskScore"]<30,

    df["FraudRiskScore"]<60,

    df["FraudRiskScore"]>=60

]

choices=[

    "Low",

    "Medium",

    "High"

]

df["CustomerRiskCategory"]=np.select(

    conditions,

    choices,

    default="Low"

)

In [ ]:
df.head()

### Encoding

In [ ]:
encoder=LabelEncoder()

for col in df.select_dtypes(include="object").columns:

    df[col]=encoder.fit_transform(df[col].astype(str))

### Scaling

In [ ]:
scaler=StandardScaler()

scaled_columns=[

    "Quantity",

    "UnitPrice",

    "ItemsInCart",

    "TotalPrice",

    "FraudRiskScore",

    "AveragePrice",

    "CustomerSpending"

]

df[scaled_columns]=scaler.fit_transform(

    df[scaled_columns]

)

### Power Transformation

In [ ]:
pt=PowerTransformer()

df[scaled_columns]=pt.fit_transform(

    df[scaled_columns]

)

In [ ]:
print(df.shape)
df.head()

### Separate Features & Target

In [ ]:
X = df.drop(columns=[
    "is_fraud",
    "FraudRiskScore",
    "CustomerRiskCategory"
], errors="ignore")

y = df["is_fraud"]

print("Feature Shape :", X.shape)
print("Target Shape :", y.shape)

print("\nFeatures Used:")
print(X.columns.tolist())

### Correlation with Target

In [ ]:
corr = df.corr(numeric_only=True)

target_corr = corr["is_fraud"].sort_values(ascending=False)

target_corr

In [ ]:
plt.figure(figsize=(8,10))

target_corr.drop("is_fraud").plot(kind="barh")

plt.title("Feature Correlation with Fraud")

plt.show()

### Variance Threshold

In [ ]:
print(X.dtypes)

In [ ]:
X = X.drop(columns=["Date"], errors="ignore")

In [ ]:
from sklearn.feature_selection import VarianceThreshold

selector = VarianceThreshold(threshold=0.01)

selector.fit(X)

selected_columns = X.columns[selector.get_support()]

print(selected_columns)

In [ ]:
X = X[selected_columns]

print(X.shape)

### Mutual Information

In [ ]:
from sklearn.feature_selection import mutual_info_classif
import pandas as pd

X_numeric = X.select_dtypes(include=["number"])

# Calculate Mutual Information
mi_scores = mutual_info_classif(
    X_numeric,
    y,
    random_state=42
)

mi = pd.DataFrame({
    "Feature": X_numeric.columns,
    "MI Score": mi_scores
})

mi = mi.sort_values(by="MI Score", ascending=False)

print(mi)

In [ ]:
plt.figure(figsize=(8,5))

sns.barplot(
    data=mi,
    x="MI Score",
    y="Feature",
    palette="viridis"
)

plt.title("Mutual Information Feature Importance")
plt.xlabel("MI Score")
plt.ylabel("Features")
plt.show()

### Random Forest Importance

In [ ]:
from sklearn.ensemble import RandomForestClassifier
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# Use only numeric features
X_rf = X.select_dtypes(include=["number"])

# Train Random Forest
rf = RandomForestClassifier(
    n_estimators=200,
    random_state=42,
    n_jobs=-1
)

rf.fit(X_rf, y)

# Feature Importance
importance = pd.DataFrame({
    "Feature": X_rf.columns,
    "Importance": rf.feature_importances_
})

importance = importance.sort_values(
    by="Importance",
    ascending=False
)

print(importance)

In [ ]:
plt.figure(figsize=(8,6))

sns.barplot(
    data=importance.head(15),
    x="Importance",
    y="Feature",
    palette="viridis"
)

plt.title("Random Forest Feature Importance")
plt.xlabel("Importance Score")
plt.ylabel("Features")

plt.show()

In [ ]:
top_features = importance.head(10)["Feature"].tolist()

print("Top 10 Features:")
print(top_features)

In [ ]:
 #X = X[top_features]

In [ ]:
from sklearn.inspection import permutation_importance

perm = permutation_importance(

    rf,

    X,

    y,

    random_state=42,

    scoring="f1"

)

perm_importance = pd.Series(

    perm.importances_mean,

    index=X.columns

)

perm_importance.sort_values(

    ascending=False

).head(20)

In [ ]:
plt.figure(figsize=(10,6))

perm_importance.sort_values(

    ascending=False

).head(20).plot(

    kind="bar"

)

plt.title("Permutation Importance")

plt.show()

### Recursive Feature Elimination

In [ ]:
from sklearn.feature_selection import RFE

rfe = RFE(

    estimator=RandomForestClassifier(),

    n_features_to_select=10

)

rfe.fit(X,y)

selected = X.columns[rfe.support_]

selected

In [ ]:
X = X[selected]

print(X.shape)

### Train Test Split

In [ ]:
from sklearn.model_selection import train_test_split

X_train,X_test,y_train,y_test = train_test_split(

    X,

    y,

    test_size=0.20,

    stratify=y,

    random_state=42

)

print(X_train.shape)

print(X_test.shape)

In [ ]:
print(y_train.value_counts())

In [ ]:
print(X_train.dtypes)

In [ ]:
X_train["WeekOfYear"] = X_train["WeekOfYear"].astype("int64")
X_test["WeekOfYear"] = X_test["WeekOfYear"].astype("int64")

### SMOTE

In [ ]:
from imblearn.over_sampling import SMOTE

smote = SMOTE(

    random_state=42

)

X_smote,y_smote = smote.fit_resample(

    X_train,

    y_train

)

print(y_smote.value_counts())

### Random Oversampling

In [ ]:
from imblearn.over_sampling import RandomOverSampler

ros = RandomOverSampler(

    random_state=42

)

X_ros,y_ros = ros.fit_resample(

    X_train,

    y_train

)

print(y_ros.value_counts())

### Random Undersampling

In [ ]:
from imblearn.under_sampling import RandomUnderSampler

rus = RandomUnderSampler(

    random_state=42

)

X_rus,y_rus = rus.fit_resample(

    X_train,

    y_train

)

print(y_rus.value_counts())

In [ ]:
from imblearn.combine import SMOTEENN

smoteenn = SMOTEENN(

    random_state=42

)

X_smoteenn,y_smoteenn = smoteenn.fit_resample(

    X_train,

    y_train

)

print(y_smoteenn.value_counts())

### Comparison Table

In [ ]:
comparison = pd.DataFrame({

    "Method":[

        "Original",

        "Random Oversampling",

        "Random Undersampling",

        "SMOTE",

        "SMOTEENN"

    ],

    "Samples":[

        len(y_train),

        len(y_ros),

        len(y_rus),

        len(y_smote),

        len(y_smoteenn)

    ]

})

comparison

In [ ]:
plt.figure(figsize=(10,5))

sns.barplot(

    x="Method",

    y="Samples",

    data=comparison

)

plt.xticks(rotation=45)

plt.title("Comparison of Sampling Techniques")

plt.show()

In [ ]:
print("""
Sampling Comparison

Original Dataset :
Highly Imbalanced

Random Oversampling :
May lead to overfitting.

Random Undersampling :
Can lose valuable information.

SMOTE :
Creates realistic synthetic samples.

SMOTEENN :
Combines oversampling and cleaning.

Recommended Method :
SMOTEENN
""")

### Save Final Dataset

In [ ]:
train_data = pd.concat(

    [

        pd.DataFrame(X_smoteenn),

        pd.Series(y_smoteenn,name="is_fraud")

    ],

    axis=1

)

train_data.head()

In [ ]:
X = X.drop(columns=["OrderID", "CustomerID"], errors="ignore")

In [ ]:
### Prepare Training Data
### Using SMOTEENN Dataset

X_train = X_smoteenn
y_train = y_smoteenn

print(X_train.shape)
print(y_train.shape)

### Evaluation Function

In [ ]:
results = []

def evaluate_model(model, name):

    model.fit(X_train, y_train)

    y_pred = model.predict(X_test)

    if hasattr(model, "predict_proba"):
        y_prob = model.predict_proba(X_test)[:,1]
    else:
        y_prob = None

    accuracy = accuracy_score(y_test, y_pred)

    precision = precision_score(y_test, y_pred)

    recall = recall_score(y_test, y_pred)

    f1 = f1_score(y_test, y_pred)

    balanced = balanced_accuracy_score(y_test, y_pred)

    mcc = matthews_corrcoef(y_test, y_pred)

    roc = roc_auc_score(y_test, y_prob) if y_prob is not None else np.nan

    results.append([
        name,
        accuracy,
        precision,
        recall,
        f1,
        balanced,
        roc,
        mcc
    ])

    print(name)

    print(classification_report(y_test,y_pred))

In [ ]:
lr = LogisticRegression(
    random_state=42,
    max_iter=1000
)

evaluate_model(
    lr,
    "Logistic Regression"
)

In [ ]:
dt = DecisionTreeClassifier(
    random_state=42
)

evaluate_model(
    dt,
    "Decision Tree"
)

In [ ]:
rf = RandomForestClassifier(
    n_estimators=200,
    random_state=42
)

evaluate_model(
    rf,
    "Random Forest"
)

In [ ]:
knn = KNeighborsClassifier(
    n_neighbors=5
)

evaluate_model(
    knn,
    "KNN"
)

In [ ]:
nb = GaussianNB()

evaluate_model(
    nb,
    "Naive Bayes"
)

### Performance Table

In [ ]:
performance = pd.DataFrame(
    results,
    columns=[
        "Model",
        "Accuracy",
        "Precision",
        "Recall",
        "F1",
        "Balanced Accuracy",
        "ROC AUC",
        "MCC"
    ]
)

performance

### Sort by F1

In [ ]:
performance.sort_values(
    by="F1",
    ascending=False
)

In [ ]:
plt.figure(figsize=(12,5))

sns.barplot(
    x="Model",
    y="F1",
    data=performance
)

plt.title("Model Comparison (F1 Score)")

plt.xticks(rotation=20)

plt.show()

In [ ]:
plt.figure(figsize=(12,5))

sns.barplot(
    x="Model",
    y="Accuracy",
    data=performance
)

plt.xticks(rotation=20)

plt.title("Accuracy Comparison")

plt.show()

In [ ]:
plt.figure(figsize=(8,6))

for model, name in zip(
    [lr,dt,rf,knn,nb],
    [
        "LR",
        "DT",
        "RF",
        "KNN",
        "NB"
    ]
):

    if hasattr(model,"predict_proba"):

        prob = model.predict_proba(X_test)[:,1]

        fpr,tpr,_ = roc_curve(y_test,prob)

        plt.plot(
            fpr,
            tpr,
            label=name
        )

plt.plot([0,1],[0,1],"k--")

plt.xlabel("False Positive Rate")

plt.ylabel("True Positive Rate")

plt.title("ROC Curve Comparison")

plt.legend()

plt.show()

In [ ]:
best = performance.sort_values(
    by="F1",
    ascending=False
)

print(best.head())

In [ ]:
# Import Advanced ML Libraries
from sklearn.svm import SVC
from sklearn.ensemble import GradientBoostingClassifier
from xgboost import XGBClassifier


In [ ]:
svm = SVC(
    kernel="rbf",
    probability=True,
    random_state=42
)

evaluate_model(
    svm,
    "Support Vector Machine"
)

In [ ]:
gb = GradientBoostingClassifier(

    random_state=42,

    n_estimators=150,

    learning_rate=0.1,

    max_depth=3

)

evaluate_model(

    gb,

    "Gradient Boosting"

)

In [ ]:
xgb = XGBClassifier(

    random_state=42,

    n_estimators=200,

    learning_rate=0.05,

    max_depth=5,

    subsample=0.8,

    colsample_bytree=0.8,

    eval_metric="logloss"

)

evaluate_model(

    xgb,

    "XGBoost"

)

In [ ]:
performance = pd.DataFrame(

    results,

    columns=[

        "Model",

        "Accuracy",

        "Precision",

        "Recall",

        "F1",

        "Balanced Accuracy",

        "ROC AUC",

        "MCC"

    ]

)

performance

### Sort by F1 Score

In [ ]:
performance = performance.sort_values(

    by="F1",

    ascending=False

)

performance

In [ ]:
plt.figure(figsize=(12,6))

sns.barplot(

    x="Model",

    y="F1",

    data=performance

)

plt.xticks(rotation=30)

plt.title("F1 Score Comparison")

plt.show()

In [ ]:
plt.figure(figsize=(8,6))

advanced_models = [

    svm,

    gb,

    xgb

]

advanced_names = [

    "SVM",

    "Gradient Boosting",

    "XGBoost"

]

for model,name in zip(

    advanced_models,

    advanced_names

):

    prob = model.predict_proba(

        X_test

    )[:,1]

    fpr,tpr,_ = roc_curve(

        y_test,

        prob

    )

    plt.plot(

        fpr,

        tpr,

        linewidth=2,

        label=name

    )

plt.plot(

    [0,1],

    [0,1],

    "k--"

)

plt.xlabel("False Positive Rate")

plt.ylabel("True Positive Rate")

plt.title("ROC Curve")

plt.legend()

plt.show()

In [ ]:
pred = xgb.predict(X_test)

cm = confusion_matrix(

    y_test,

    pred

)

plt.figure(figsize=(6,5))

sns.heatmap(

    cm,

    annot=True,

    fmt="d",

    cmap="Blues"

)

plt.xlabel("Predicted")

plt.ylabel("Actual")

plt.title("XGBoost Confusion Matrix")

plt.show()

In [ ]:
print(

classification_report(

    y_test,

    pred

)

)

In [ ]:
plt.figure(figsize=(12,5))

sns.barplot(

    x="Model",

    y="ROC AUC",

    data=performance

)

plt.xticks(rotation=25)

plt.title("ROC-AUC Comparison")

plt.show()

In [ ]:
plt.figure(figsize=(12,5))

sns.barplot(

    x="Model",

    y="Balanced Accuracy",

    data=performance

)

plt.xticks(rotation=25)

plt.title("Balanced Accuracy")

plt.show()

In [ ]:
best_model = performance.iloc[0]

print("Best Performing Model")

print(best_model)

In [ ]:
performance.to_csv(

    "Model_Comparison.csv",

    index=False

)

print("Model Comparison Saved")

### Voting Classifier

In [ ]:
from sklearn.ensemble import VotingClassifier

voting = VotingClassifier(
    estimators=[
        ("lr", lr),
        ("rf", rf),
        ("xgb", xgb)
    ],
    voting="soft"
)

evaluate_model(voting, "Voting Classifier")

### Stacking Classifier

In [ ]:
from sklearn.ensemble import StackingClassifier
from sklearn.linear_model import LogisticRegression

stack = StackingClassifier(
    estimators=[
        ("rf", rf),
        ("xgb", xgb),
        ("gb", gb)
    ],
    final_estimator=LogisticRegression(),
    cv=5
)

evaluate_model(stack, "Stacking Classifier")

In [ ]:
performance = pd.DataFrame(
    results,
    columns=[
        "Model",
        "Accuracy",
        "Precision",
        "Recall",
        "F1",
        "Balanced Accuracy",
        "ROC AUC",
        "MCC"
    ]
)

performance.sort_values("F1", ascending=False)

In [ ]:
best_model = performance.sort_values(
    "F1",
    ascending=False
).iloc[0]

print(best_model)

In [ ]:
import joblib

joblib.dump(xgb, "Best_Fraud_Model.pkl")

print("Model Saved Successfully")

### Sklearn Pipeline

In [ ]:
from sklearn.pipeline import Pipeline

pipeline = Pipeline([
    ("scaler", StandardScaler()),
    ("model", xgb)
])

pipeline.fit(X_train, y_train)

In [ ]:
pred = pipeline.predict(X_test)

print(classification_report(y_test, pred))

### GridSearchCV

In [ ]:
from sklearn.model_selection import GridSearchCV

param_grid = {
    "n_estimators":[100,200],
    "max_depth":[3,5,7],
    "learning_rate":[0.01,0.1]
}

grid = GridSearchCV(
    XGBClassifier(random_state=42, eval_metric="logloss"),
    param_grid,
    cv=5,
    scoring="f1",
    n_jobs=-1
)

grid.fit(X_train,y_train)

print(grid.best_params_)

### RandomizedSearchCV

In [ ]:
from sklearn.model_selection import RandomizedSearchCV

params = {
    "n_estimators":[100,150,200,250],
    "max_depth":[3,5,7,9],
    "learning_rate":[0.01,0.05,0.1],
    "subsample":[0.7,0.8,1]
}

random_search = RandomizedSearchCV(
    XGBClassifier(random_state=42, eval_metric="logloss"),
    params,
    cv=5,
    scoring="f1",
    n_iter=10,
    random_state=42
)

random_search.fit(X_train,y_train)

print(random_search.best_params_)

### Cross Validation

In [ ]:
from sklearn.model_selection import cross_val_score

scores = cross_val_score(
    xgb,
    X_train,
    y_train,
    cv=5,
    scoring="f1"
)

print(scores)
print(scores.mean())

In [ ]:
best_model = grid.best_estimator_

best_model.fit(X_train,y_train)

pred = best_model.predict(X_test)

prob = best_model.predict_proba(X_test)[:,1]

### Evaluation

In [ ]:
from sklearn.metrics import *

print(classification_report(y_test,pred))

print("Accuracy :",accuracy_score(y_test,pred))
print("Precision :",precision_score(y_test,pred))
print("Recall :",recall_score(y_test,pred))
print("F1 :",f1_score(y_test,pred))
print("ROC :",roc_auc_score(y_test,prob))
print("Balanced Accuracy :",balanced_accuracy_score(y_test,pred))
print("MCC :",matthews_corrcoef(y_test,pred))

In [ ]:
cm = confusion_matrix(y_test,pred)

plt.figure(figsize=(5,4))

sns.heatmap(cm,
            annot=True,
            fmt="d",
            cmap="Blues")

plt.title("Confusion Matrix")

plt.show()

In [ ]:
fpr,tpr,_ = roc_curve(y_test,prob)

plt.figure(figsize=(6,5))

plt.plot(fpr,tpr,label="ROC")

plt.plot([0,1],[0,1],"k--")

plt.legend()

plt.show()

In [ ]:
from sklearn.metrics import precision_recall_curve

p,r,_ = precision_recall_curve(y_test,prob)

plt.figure(figsize=(6,5))

plt.plot(r,p)

plt.title("Precision Recall Curve")

plt.show()

In [ ]:
thresholds=np.arange(0.1,0.95,0.05)

best_threshold=0.5
best_f1=0

for t in thresholds:

    pred_new=(prob>=t).astype(int)

    score=f1_score(y_test,pred_new)

    if score>best_f1:

        best_f1=score

        best_threshold=t

print(best_threshold,best_f1)

In [ ]:
from sklearn.model_selection import learning_curve

train_sizes,train_scores,test_scores = learning_curve(
    xgb,
    X_train,
    y_train,
    cv=5
)

plt.plot(train_sizes,
         train_scores.mean(axis=1),
         label="Train")

plt.plot(train_sizes,
         test_scores.mean(axis=1),
         label="Validation")

plt.legend()

plt.show()

In [ ]:
importance = pd.Series(
    best_model.feature_importances_,
    index=X_train.columns
)

importance.sort_values().tail(15).plot(
    kind="barh",
    figsize=(8,6)
)

plt.show()

In [ ]:
errors = X_test.copy()

errors["Actual"] = y_test.values

errors["Predicted"] = pred

misclassified = errors[
    errors["Actual"] != errors["Predicted"]
]

misclassified.head()

In [ ]:
import joblib

joblib.dump(best_model,"Fraud_Detection_Model.pkl")

print("Project Completed Successfully")

In [ ]:
print(X.columns)

In [ ]:
print(y.value_counts())

In [ ]:
print(y.value_counts(normalize=True) * 100)

# Important
### Although several models achieved similar Accuracy and F1-score, Random Forest was selected as the final model because it achieved the highest ROC-AUC score (0.9886) while maintaining perfect Recall (1.00). In fraud detection, minimizing False Negatives is critical because missing a fraudulent transaction can lead to direct financial losses. Therefore, a model with high Recall and strong ROC-AUC was preferred over one with marginally higher Accuracy.